# Reproducing the paper results

This notebook walks through the quantitative results of the paper one by one, using the kernel provided in `src/`. Run cells in order.

**Prerequisites:**
- Environment installed from `requirements.txt`
- The state cache built once (see cell 2)

**Estimated time to run this notebook end to end:** ~15 minutes (skipping the long parallel experiments; those are launched from `scripts/`, not from here).

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Add src/ to the Python path so we can import the kernel.
# This notebook lives in notebooks/, so src is at ../src
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pickle
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ca_kernel import run_one, run_fete, build_state

print(f'Repo root: {REPO_ROOT}')

## 1. Build the state (one-time)

The state object encapsulates: the four normalised input grids (h̃, p̃, W, T), the derived surfaces (E, c), the 8-neighbour graph with base weights, the extracted attractors, and the site mask.

This takes ~30 seconds. If `state_cache.pkl` already exists in the repo root, we load it instead.

In [ ]:
state_path = REPO_ROOT / 'state_cache.pkl'
csv_path   = REPO_ROOT / 'data' / 'grilla_con_W_hidrica.csv'

if state_path.exists():
    print(f'Loading existing state from {state_path}')
    with open(state_path, 'rb') as f:
        state = pickle.load(f)
else:
    print(f'Building state from {csv_path} (takes ~30s)')
    state = build_state(str(csv_path))
    with open(state_path, 'wb') as f:
        pickle.dump(state, f)

print(f'Sites          : {state.n_sites}')
print(f'Attractors     : {len(state.attractors)}')
print(f'Attractor pairs: {len(state.pair_candidates)}')
print(f'Graph nodes    : {state.G_template.number_of_nodes()}')
print(f'Graph edges    : {state.G_template.number_of_edges()}')

**Expected output:** 84 sites, 24 attractors, 86 pairs, 12826 nodes, ~50600 edges. This matches Table 1 of the paper.

---

## 2. Canonical CA run (§4.1)

The canonical configuration is ε = 0.15, κ = 0.03. One run takes ~7 minutes on a single CPU.

In [ ]:
t0 = time.time()
result = run_one(state, eps=0.15, kappa=0.03, seed=42)
print(f'Runtime : {time.time()-t0:.1f}s')
print(f'AUC     : {result["auc"]:.4f}')
print(f'LCC size: {result["lcc_size"]}')
print(f'm1      : {result["m1"]:.4f}')
print(f'm2      : {result["m2"]:.4f}')
print(f'm3      : {result["m3"]:.4f}')

**Expected values (paper canonical):** AUC ≈ 0.816, LCC ≈ 138, m1 ≈ 0.82, m2 ≈ 0.35.

---

## 3. FETE baseline (§4.5, Table 3)

The FETE baseline computes a single deterministic least-cost path between random endpoint pairs. Fast: ~10 seconds per run at n_pairs=86.

Here we run 5 seeds at matched budget (n_pairs=86) and compare against the CA canonical value from cell 3.

In [ ]:
fete_results = []
for seed in range(5):
    r = run_fete(state, n_random_pairs=86, seed=seed)
    fete_results.append(r)
    print(f'seed={seed}: AUC={r["auc"]:.4f}, LCC={r["lcc_size"]}, m2={r["m2"]:.4f}')

df = pd.DataFrame(fete_results)
print()
print(f'FETE mean  : AUC = {df["auc"].mean():.4f} ± {df["auc"].std():.4f}')
print(f'FETE LCC   : {df["lcc_size"].mean():.1f} ± {df["lcc_size"].std():.1f}')
print(f'FETE m2    : {df["m2"].mean():.4f} ± {df["m2"].std():.4f}')

**Expected values from paper (30 seeds):**
- CA canonical: AUC = 0.816 ± 0.002, LCC = 138 ± 23, m2 = 0.348 ± 0.061
- FETE (n=86): AUC = 0.810 ± 0.008, LCC = 146 ± 20, m2 = 0.27 ± 0.05

With only 5 seeds you will see slightly wider error bars. For the full statistics (30 seeds), run `scripts/run_fete_local.py` (~1 hour on 7 workers).

---

## 4. KS diagnostic (§4.3)

Kolmogorov–Smirnov two-sample test on L values at sites vs. background.

In [ ]:
from scipy.stats import ks_2samp

# Rebuild the L field for the canonical seed (uses cached seed=42 result if same)
from ca_kernel import build_fixation_field

# The run_one call above already computed the L field internally; here we
# re-derive it explicitly for the KS test.
# (For the paper we use the analyzer at src/analyze/analyze_ks_test.py which
#  packages this in a reusable form. Here we just show the core computation.)

# Recompute L quickly
print('Re-running canonical seed=42 for KS analysis...')
r = run_one(state, eps=0.15, kappa=0.03, seed=42, return_fields=True)
L_field = r['L']
valid = state.valid_mask
site_mask = state.site_mask

# Full-site KS
L_at_sites = L_field[valid & site_mask]
L_at_bg    = L_field[valid & ~site_mask]
D, p = ks_2samp(L_at_sites, L_at_bg)
print(f'\nFull sites (n={site_mask.sum()}):')
print(f'  D = {D:.4f}, p = {p:.2e}')

**Expected values:**
- Full sites, L: D = 0.543, p = 1.87e-23

For the full leave-dominant-out variant reported in the paper, run `python src/analyze/analyze_ks_test.py --state state_cache.pkl` from the shell.

---

## 5. Zonal Spearman correlation (§4.4)

Partition the valid cells into 20 K-means zones over the standardised feature vector (x, y, h̃, p̃), then compute Spearman ρ between per-zone mean L and per-zone site count.

In [ ]:
from sklearn.cluster import KMeans
from scipy.stats import spearmanr

valid = state.valid_mask
yy, xx = np.where(valid)
h_at   = state.h_grid[valid]
p_at   = state.p_grid[valid]

X = np.column_stack([xx, yy, h_at, p_at]).astype(float)
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)

km = KMeans(n_clusters=20, n_init=10, random_state=0)
labels = km.fit_predict(X)

L_at_valid   = L_field[valid]
site_at_valid = site_mask[valid].astype(int)

L_mean_per_zone   = np.array([L_at_valid[labels == z].mean() for z in range(20)])
sites_per_zone    = np.array([site_at_valid[labels == z].sum() for z in range(20)])

rho, p_rho = spearmanr(L_mean_per_zone, sites_per_zone)
print(f'Zonal Spearman correlation (n=20): rho = {rho:+.3f}, p = {p_rho:.2e}')

**Expected value:** ρ ≈ +0.55, p < 0.05.

---

## Next steps

For the parallel experiments that produce the paper's aggregate statistics, use the scripts in `scripts/`:

- `python scripts/run_fete_local.py`         — full FETE baseline (~1 h)
- `python scripts/run_ablation_weights.py`   — weight ablation (~6 h)
- `python scripts/run_phase1_local.py`       — (ε, κ) phase scan (~30 h)

For figure regeneration:

- `python scripts/regen_figure2.py`
- `python scripts/regen_figures_with_scalebar.py`

See `docs/REPRODUCE.md` for the full guide.